SETUP

In [1]:
import pandas as pd
import sqlite3
import openpyxl

In [2]:
mer_df = pd.read_csv("csv_file/merchants.csv")
refund_df = pd.read_csv("csv_file/refunds.csv")
tran_info_df = pd.read_csv("csv_file/transaction_info.csv")
tran_df = pd.read_csv("csv_file/transactions.csv")

In [3]:
display(mer_df[:1])
display(refund_df[:1])
display(tran_info_df[:1])
display(tran_df[:1])

,merchant_id,leader_name,create_date,address,country,email,phone,industry
0,"Walker, Floyd and Graham",Keith Shelton,2025-01-04,"87980 Andrew Parks, Brandonfurt, NY 92602",United States,edwardadams@bradford-drake.org,(569)442-6080,Health & Beauty


,refund_id,tran_id,amount,date,reason,status
0,deda1a7b-e213-4df0-82cf-b79f34b7fa0d,16030122-4b72-49bc-926b-dfcb25434fd0,11.36,2023-10-18 03:31:36,Product Defect,completed


,tran_id,country,card_type,card_brands,region,channel_type
0,5f957278-bcfc-4295-96dc-e2d785f053aa,Malaysia,NaN,NaN,Domestic,Buy Now Pay Later


,tran_id,merchant_id,channel,amount,currency,date,status
0,5f957278-bcfc-4295-96dc-e2d785f053aa,Christensen-Cole,PayLater by Grab,38.07,MYR,2023-02-03 01:37:24,completed


In [4]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Load DataFrames into SQL tables
mer_df.to_sql("merchants", conn, index=False, if_exists='replace')
refund_df.to_sql("refunds", conn, index=False, if_exists='replace')
tran_info_df.to_sql("transaction_info", conn, index=False, if_exists='replace')
tran_df.to_sql("transactions", conn, index=False, if_exists='replace')

545668

GLOBAL FUNCTION

In [5]:
def convert_currency_to_MYR(dataframe):
    rates = {
        "MYR": 1,
        "SGD": 0.3042,
        "THB": 7.5380,
        "USD": 0.2372,
        "GBP": 0.1753,
        "CNY": 1.6897
    }

    dataframe = dataframe.copy()

    dataframe["amount_MYR"] = round(dataframe.apply(
        lambda x: x["amount"] / rates.get(x["currency"], 1), axis=1
    ), 2)
    
    return dataframe

In [6]:
def apply_refund_rate_lvl(rate):
    if rate > 6:
        return 'high'
    elif 5 < rate <= 6:
        return 'average'
    else:
        return 'low'

QUERY

In [ ]:
sqlInput = """
SELECT merchant_id, country, email, phone
FROM merchants
WHERE country IN ('Malaysia', 'China', 'Singapore', 'Thailand')
"""

asiaMer = pd.read_sql_query(sqlInput, conn)

display(asiaMer)

asiaMer_list = tuple(asiaMer['merchant_id'].tolist())
print(len(asiaMer_list))

,merchant_id,country,email,phone
0,Landry-Moreno,China,katherinelarson@murray-morgan.com,320.353.9180x5315
1,"Ingram, Pratt and York",China,jhayes@griffin.com,001-696-491-6581
2,Reid Group,Singapore,psparks@howe.org,490-777-3437x479
3,Young-Huffman,Malaysia,smithjade@khan.com,001-420-934-1353x54055
4,Johnson Inc,Singapore,millerhaley@peters.com,881.877.3747
...,...,...,...,...
60,Martinez-Harvey,Malaysia,emiller@white.com,507-721-5280x99799
61,"Maynard, Green and Cook",Singapore,watsonsydney@hernandez.com,001-371-857-9186
62,Jordan Ltd,Singapore,jjackson@reynolds.com,+1-285-417-2466x765
63,"Schmitt, Murphy and Warner",China,longjose@leach-downs.info,285.409.6910x57000


65


In [30]:
sqlInput = f"""
SELECT t.tran_id, t.status, t.date, t.merchant_id, t.channel, t.amount, t.currency, 
    ti.card_type, ti.channel_type, 
    COUNT(DISTINCT r.tran_id) AS refund_count, SUM(r.amount) AS refund_amt, r.reason
FROM transactions t
    LEFT JOIN transaction_info ti on t.tran_id = ti.tran_id
    LEFT JOIN refunds r on t.tran_id = r.tran_id AND r.status = 'completed'
WHERE t.merchant_id IN {asiaMer_list}
    AND t.status IN ('completed', 'recorded', 'cancelled')
    AND t.date BETWEEN '2024-01-01 00:00:00' AND '2024-12-31 23:59:59'
GROUP BY t.tran_id
"""

asiaTran = pd.read_sql_query(sqlInput, conn)

display(asiaTran)

,tran_id,status,date,merchant_id,channel,amount,currency,card_type,channel_type,refund_count,refund_amt,reason
0,000095eb-14b1-4886-897e-f0161e3cc40d,cancelled,2024-02-03 12:54:29,"Jones, Cole and Haynes",Google Pay,9.22,MYR,credit,Credit Card Online,1,9.22,Customer Dissatisfaction
1,00015b22-f9a3-40ab-8c42-50f0d60734f7,completed,2024-04-28 06:16:34,Jordan Ltd,TNG eWallet,41.07,MYR,None,E-wallet,0,NaN,None
2,0001ee9d-7936-4ddb-abcd-b6f9836e4686,completed,2024-02-23 17:23:01,Nolan LLC,Apple Pay,12.63,MYR,debit,Credit Card Online,0,NaN,None
3,0001f804-3823-4f7a-a05b-6e0d2c103fa7,completed,2024-01-02 13:42:03,"Johnson, Moore and Smith",FPX,197.60,USD,None,Online Bank,0,NaN,None
4,000260a6-ed32-465d-983d-c5d5e77c9a7a,completed,2024-02-14 10:47:06,"Johnson, Moore and Smith",FPX,29.47,MYR,None,Online Bank,0,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...
103572,fffc024e-d8ce-4db5-8b31-50428808ecaf,cancelled,2024-12-30 04:48:26,Gross-Harrington,TNG eWallet,32.42,MYR,None,E-wallet,1,32.42,Wrong Item Sent
103573,fffc2ada-dbeb-479c-a735-1e0d05157661,completed,2024-07-28 14:18:35,"Haynes, Jenkins and Cook",Visa,57.32,MYR,debit,Credit Card Offline,0,NaN,None
103574,fffdcdfd-10f9-493a-a819-d3553b36858d,completed,2024-09-11 16:53:39,Russell and Sons,TNG eWallet Offline,1.09,THB,None,E-wallet Offline,0,NaN,None
103575,fffe49bc-d86f-4001-9cd0-20658f80748a,completed,2024-07-11 11:03:59,Johnson Inc,FPX,53.88,SGD,None,Online Bank,0,NaN,None


In [46]:
asiaTran['refund_amt'] = asiaTran['refund_amt'].fillna(0)
converted_asiaTran = convert_currency_to_MYR(asiaTran)
converted_asiaTran = converted_asiaTran.rename(columns={'amount':'txn_amt', 'amount_MYR':'txn_amt_MYR', 'refund_amt':'amount'})
converted_asiaTran = convert_currency_to_MYR(converted_asiaTran)
converted_asiaTran = converted_asiaTran.rename(columns={'amount_MYR':'refund_amt_MYR'})
display(converted_asiaTran[converted_asiaTran['refund_amt_MYR'] != 0][:5])

,tran_id,status,date,merchant_id,channel,txn_amt,currency,card_type,channel_type,refund_count,amount,reason,txn_amt_MYR,refund_amt_MYR
0,000095eb-14b1-4886-897e-f0161e3cc40d,cancelled,2024-02-03 12:54:29,"Jones, Cole and Haynes",Google Pay,9.22,MYR,credit,Credit Card Online,1,9.22,Customer Dissatisfaction,9.22,9.22
26,000e383b-beee-4d48-bf8b-d8d10b60b5bc,cancelled,2024-09-16 07:10:23,Gross-Harrington,TNG eWallet Offline,22.69,MYR,None,E-wallet Offline,1,22.69,Delayed Delivery,22.69,22.69
30,0010b974-3e71-4029-bc7c-04b555774b48,cancelled,2024-10-14 11:22:18,Jones LLC,TNG eWallet Offline,49.06,GBP,None,E-wallet Offline,1,34.21,Fraudulent Transaction,279.86,195.15
47,0018fb6d-b682-4501-a4f7-079369270d4d,cancelled,2024-04-28 14:43:15,Martinez-Harvey,DuitNow QR,56.45,MYR,None,E-wallet,1,56.45,Wrong Item Sent,56.45,56.45
66,00261811-d6a6-44a5-a7b3-f0b5b8f70181,completed,2024-05-23 10:28:00,Christensen-Cole,PayLater by Grab,32.31,MYR,None,Buy Now Pay Later,1,32.31,Delayed Delivery,32.31,32.31


In [47]:
converted_asiaTran["partial_refund_count"] = converted_asiaTran.apply(
    lambda x: 1 if (x["txn_amt_MYR"] > 0 and 0 < x["refund_amt_MYR"] < x["txn_amt_MYR"]) else 0,
    axis=1
)

converted_asiaTran["fully_refund_count"] = converted_asiaTran.apply(
    lambda x: 1 if (x["txn_amt_MYR"] > 0 and x["refund_amt_MYR"] == x["txn_amt_MYR"]) else 0,
    axis=1
)

grouped_asiaTran = converted_asiaTran.groupby(["merchant_id"]).agg(
    channel=('channel', lambda x: ', '.join(map(str, x.unique()))),
    txn_count=('status', lambda x: x.isin(['completed', 'recorded']).sum()),
    refund_count=('refund_count', 'sum'),
    txn_amt=('txn_amt_MYR', 'sum'),
    refund_amt=('refund_amt_MYR', 'sum'),
    partial_refund_count=('partial_refund_count', 'sum'),
    fully_refund_count=('fully_refund_count', 'sum')
).reset_index()

display(grouped_asiaTran[:5])

,merchant_id,channel,txn_count,refund_count,txn_amt,refund_amt,partial_refund_count,fully_refund_count
0,Aguilar Group,"TNG eWallet, DuitNow QR, DuitNow QR Offline, V...",2556,138,248985.47,10150.69,21,117
1,"Aguilar, Bonilla and Lee","DuitNow QR, DuitNow QR Offline, Shopee PayLate...",1280,75,132570.31,10561.60,14,61
2,Bender-Underwood,"DuitNow QR Offline, Visa, DuitNow QR, FPX, Mas...",1326,77,140023.13,6490.34,14,63
3,Cannon and Sons,"TNG eWallet Offline, DuitNow QR Offline, FPX, ...",1506,93,143613.50,7884.66,11,82
4,Christensen-Cole,"DuitNow QR Offline, PayLater by Grab, TNG eWal...",1926,110,176843.14,10240.53,13,97


In [48]:
grouped_asiaTran['refund_rate'] = round((grouped_asiaTran['refund_count']/grouped_asiaTran['txn_count'])*100, 2)
grouped_asiaTran['partial_refund_rate'] = round((grouped_asiaTran['partial_refund_count']/grouped_asiaTran['refund_count'])*100, 2)
grouped_asiaTran['fully_refund_rate'] = round((grouped_asiaTran['fully_refund_count']/grouped_asiaTran['refund_count'])*100, 2)
grouped_asiaTran['refund_lvl'] = grouped_asiaTran['refund_rate'].apply(apply_refund_rate_lvl)
display(grouped_asiaTran[:5])

,merchant_id,channel,txn_count,refund_count,txn_amt,refund_amt,partial_refund_count,fully_refund_count,refund_rate,partial_refund_rate,fully_refund_rate,refund_lvl
0,Aguilar Group,"TNG eWallet, DuitNow QR, DuitNow QR Offline, V...",2556,138,248985.47,10150.69,21,117,5.40,15.22,84.78,average
1,"Aguilar, Bonilla and Lee","DuitNow QR, DuitNow QR Offline, Shopee PayLate...",1280,75,132570.31,10561.60,14,61,5.86,18.67,81.33,average
2,Bender-Underwood,"DuitNow QR Offline, Visa, DuitNow QR, FPX, Mas...",1326,77,140023.13,6490.34,14,63,5.81,18.18,81.82,average
3,Cannon and Sons,"TNG eWallet Offline, DuitNow QR Offline, FPX, ...",1506,93,143613.50,7884.66,11,82,6.18,11.83,88.17,high
4,Christensen-Cole,"DuitNow QR Offline, PayLater by Grab, TNG eWal...",1926,110,176843.14,10240.53,13,97,5.71,11.82,88.18,average


In [49]:
asia_merged = pd.merge(asiaMer, grouped_asiaTran, on='merchant_id', how='left')
asia_merged = asia_merged.sort_values(by=['refund_rate'], ascending=[False] )
display(asia_merged)

,merchant_id,country,email,phone,channel,txn_count,refund_count,txn_amt,refund_amt,partial_refund_count,fully_refund_count,refund_rate,partial_refund_rate,fully_refund_rate,refund_lvl
5,Williams Inc,Malaysia,tylerjose@kelley.com,001-907-594-7730,"DuitNow QR Offline, TNG eWallet, FPX, DuitNow ...",1191,85,103420.45,6640.78,12,73,7.14,14.12,85.88,high
51,Phillips LLC,Thailand,nkennedy@wright-patterson.com,+1-745-250-0454x7427,"FPX, TNG eWallet Offline, Mastercard, Visa, Du...",801,56,72713.88,4004.14,11,45,6.99,19.64,80.36,high
41,"Knox, Bullock and Bowman",Thailand,xfreeman@roberts.com,947.269.5381x133,"FPX, TNG eWallet Offline, DuitNow QR Offline, ...",1848,127,188133.43,7846.33,23,104,6.87,18.11,81.89,high
9,King-Friedman,China,carterlaurie@jones.com,001-764-216-7393x6254,"DuitNow QR Offline, DuitNow QR, Visa, TNG eWal...",1991,134,190473.74,10193.77,21,113,6.73,15.67,84.33,high
32,Rollins-Williams,China,ftaylor@mitchell.org,601-998-6634x3660,"DuitNow QR, FPX, Shopee PayLater, TNG eWallet,...",750,50,77268.12,3691.05,7,43,6.67,14.00,86.00,high
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43,"Williamson, Jones and Higgins",Thailand,wayneroberts@diaz-cruz.com,778-213-9368x784,"TNG eWallet Offline, DuitNow QR, Apple Pay, TN...",677,34,61517.41,2852.48,3,31,5.02,8.82,91.18,average
26,"Gonzalez, Klein and French",Thailand,ramosalicia@reed.com,001-311-701-8511x4555,"TNG eWallet Offline, Apple Pay, Google Pay, TN...",1633,78,178048.76,6050.04,13,65,4.78,16.67,83.33,low
33,"Johnson, Key and Wells",Malaysia,markstrevor@adkins.com,001-836-518-3017,"TNG eWallet, Visa, FPX, DuitNow QR, Mastercard...",1245,55,110623.25,4868.44,7,48,4.42,12.73,87.27,low
50,"Cox, Clark and Davis",Singapore,andrewsdarius@campbell.com,573.456.1994,"TNG eWallet Offline, DuitNow QR Offline, Googl...",771,34,63293.54,2775.70,2,32,4.41,5.88,94.12,low


In [50]:
asia_merged = asia_merged.drop(columns={'partial_refund_count', 'fully_refund_count'})
asia_merged['txn_amt'] = round(asia_merged['txn_amt'], 2)
asia_merged['refund_amt'] = round(asia_merged['refund_amt'], 2)
asia_merged = asia_merged.rename(columns={'merchant_id':'Merchant ID', 'country':'Country', 'email':'Email', 'phone':'Phone Number',
                                         'channel':'Channel', 'txn_count':'Transaction Count', 'refund_count':'Refund Count', 'txn_amt':'Transaction Amount (MYR)',
                                         'refund_amt':'Refund Amount (MYR)', 'refund_rate':'Refund Rate (%)', 'partial_refund_rate':'Partial Refund Rate (%)',
                                         'fully_refund_rate':'Fully Refund Rate (%)', 'refund_lvl':'Refund Level'})
display(asia_merged[:1])
asia_merged.to_csv(f'Asia_Merchant_Refund_Rate_2024.csv', index = False)

,Merchant ID,Country,Email,Phone Number,Channel,Transaction Count,Refund Count,Transaction Amount (MYR),Refund Amount (MYR),Refund Rate (%),Partial Refund Rate (%),Fully Refund Rate (%),Refund Level
5,Williams Inc,Malaysia,tylerjose@kelley.com,001-907-594-7730,"DuitNow QR Offline, TNG eWallet, FPX, DuitNow ...",1191,85,103420.45,6640.78,7.14,14.12,85.88,high


Checking

In [11]:
sqlInput = """
SELECT merchant_id, COUNT(tran_id) AS txn_count, SUM(amount) AS total_amt
FROM transactions
WHERE date BETWEEN '2025-12-31 00:00:00' AND '2025-12-31 23:59:59'
AND status IN ('completed', 'recorded')
GROUP BY merchant_id
ORDER BY txn_count desc
"""

txncheck = pd.read_sql_query(sqlInput, conn)

display(txncheck)

,merchant_id,txn_count,total_amt
0,Bender-Underwood,8,267.55
1,Wall LLC,6,270.93
2,Pruitt LLC,5,135.84
3,Moss-Chapman,5,113.96
4,Aguilar-Gomez,5,93.27
...,...,...,...
88,Cross PLC,1,9.51
89,Cannon and Sons,1,118.54
90,Arnold PLC,1,1.92
91,"Aguilar, Bonilla and Lee",1,27.43
